# PyTorch / MLX Diffusion Model Usage

이 노트북은 현재 코드베이스의 PyTorch 기반 diffusion model과 MLX 기반 diffusion model을 같은 demo dataset으로 실행하는 예제입니다.

- PyTorch: `diffusion_hash_inv.models.conditional_diffusion`
- MLX: `diffusion_hash_inv.models.conditional_diffusion_mlx`
- MLX toy: `diffusion_hash_inv.models.diffusion_with_mlx`

기본 셀은 작은 `message.png`/JSON fixture를 만든 뒤 1-step smoke training을 수행하도록 구성되어 있습니다.

## 1. Environment Setup

In [ ]:
from pathlib import Path
import json
import shutil
import sys

import numpy as np
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    for parent in ROOT.parents:
        if (parent / "pyproject.toml").exists():
            ROOT = parent
            break

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DEMO_ROOT = ROOT / "notebooks" / "_demo_diffusion_data"
IMAGE_ROOT = DEMO_ROOT / "data" / "images"
JSON_ROOT = DEMO_ROOT / "output" / "json"
PYTORCH_OUT = DEMO_ROOT / "output" / "conditional_diffusion_torch"
MLX_OUT = DEMO_ROOT / "output" / "conditional_diffusion_mlx"

ROOT, DEMO_ROOT

## 2. 현재 구현된 Diffusion Model 종류

| 구현 | Framework | 용도 |
| --- | --- | --- |
| `conditional_diffusion.py` | PyTorch | `message.png`와 최종 해시 라벨 기반 conditional DDPM |
| `unconditional_ddpm.py` | PyTorch | 라벨 없는 baseline DDPM |
| `guided_conditional_diffusion.py` | PyTorch | classifier / classifier-free guided DDPM |
| `loop_conditioned_diffusion.py` | PyTorch | MD5 loop-state tensor 조건 DDPM |
| `diffusion_with_mlx.py` | MLX | synthetic prototype 기반 toy DDPM |
| `conditional_diffusion_mlx.py` | MLX | `message.png`와 최종 해시 라벨 기반 conditional DDPM |

## 3. Demo Dataset 생성

PyTorch/MLX 실제 데이터셋 기반 모델은 아래 구조를 기대합니다.

```text
data/images/<run-id>/message.png
output/json/**/<run-id>.json
```

이 셀은 노트북 전용 작은 fixture를 만듭니다.

In [ ]:
def write_message_png(path: Path, size: tuple[int, int], color: tuple[int, int, int]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    image = Image.new("RGB", size, color)
    image.save(path)


def write_run_json(json_root: Path, run_id: str, final_hash: str) -> None:
    path = json_root / "2026-05-12 12-00-00" / f"{run_id}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(
            {
                "Message": {"Hex": "0x00"},
                "Logs": {"4th Step": {"1st Round": {"Loop Start": {"A": "0x00"}}}},
                "Generated hash": final_hash,
                "Correct   hash": final_hash,
            },
            indent=2,
        ),
        encoding="utf-8",
    )


if DEMO_ROOT.exists():
    shutil.rmtree(DEMO_ROOT)

runs = [
    ("RUN_0001", (16, 8), (240, 40, 40), "0xaaa"),
    ("RUN_0002", (8, 16), (40, 180, 240), "0xbbb"),
    ("RUN_0003", (16, 16), (60, 220, 90), "0xccc"),
    ("RUN_0004", (16, 16), (230, 210, 60), "0xddd"),
]

for run_id, size, color, final_hash in runs:
    write_message_png(IMAGE_ROOT / run_id / "message.png", size, color)
    write_run_json(JSON_ROOT, run_id, final_hash)

sorted(path.relative_to(DEMO_ROOT) for path in DEMO_ROOT.rglob("*") if path.is_file())

## 4. PyTorch Conditional DDPM 실행

아래 cell은 아주 작은 설정으로 1-step smoke training을 수행합니다. 실제 학습에서는 `train_steps`, `timesteps`, `base_channels`, `time_dim`, `image_size`를 키우면 됩니다.

In [ ]:
RUN_PYTORCH = True

if RUN_PYTORCH:
    import torch
    from diffusion_hash_inv.models.conditional_diffusion import (
        ConditionalDiffusionTrainConfig,
        train_conditional_diffusion,
    )

    torch_config = ConditionalDiffusionTrainConfig(
        data_root=IMAGE_ROOT,
        json_root=JSON_ROOT,
        output_dir=PYTORCH_OUT,
        image_size=16,
        channels=3,
        fit_mode="pad",
        batch_size=2,
        train_steps=1,
        timesteps=4,
        base_channels=8,
        time_dim=16,
        sample_count=2,
        log_every=1,
        device="cpu",
        seed=0,
    )

    torch_result = train_conditional_diffusion(torch_config)
    torch_result
else:
    print("Set RUN_PYTORCH = True to run the PyTorch example.")

## 5. MLX Conditional DDPM 실행

`conditional_diffusion_mlx.py`는 같은 fixture를 MLX array로 읽고, 최종 해시 라벨 조건으로 MLP denoiser를 학습합니다.

In [ ]:
RUN_MLX = True

if RUN_MLX:
    import mlx.core as mx
    from diffusion_hash_inv.models.conditional_diffusion_mlx import (
        MLXConditionalDiffusionTrainConfig,
        train_conditional_diffusion_mlx,
    )

    mlx_config = MLXConditionalDiffusionTrainConfig(
        data_root=IMAGE_ROOT,
        json_root=JSON_ROOT,
        output_dir=MLX_OUT,
        image_size=8,
        channels=1,
        fit_mode="pad",
        batch_size=2,
        train_steps=1,
        timesteps=4,
        time_dim=8,
        hidden_dim=16,
        sample_count=2,
        columns=2,
        log_every=1,
        device="cpu",
        seed=0,
    )

    mlx_sample_path = train_conditional_diffusion_mlx(mlx_config)
    mlx_sample_path
else:
    print("Set RUN_MLX = True to run the MLX example.")

## 6. MLX Synthetic Toy 실행

실제 이미지/JSON 데이터 없이 MLX training loop와 sampling만 빠르게 확인할 때 사용합니다.

In [ ]:
RUN_MLX_TOY = False

if RUN_MLX_TOY:
    from diffusion_hash_inv.models.diffusion_with_mlx import build_arg_parser, run_demo

    args = build_arg_parser().parse_args(
        [
            "--device", "cpu",
            "--train-steps", "1",
            "--timesteps", "2",
            "--batch-size", "2",
            "--samples", "2",
            "--columns", "2",
            "--image-size", "8",
            "--hidden-dim", "16",
            "--time-dim", "8",
            "--output", str(DEMO_ROOT / "output" / "mlx_toy_samples.png"),
        ]
    )
    toy_output = run_demo(args)
    toy_output
else:
    print("Set RUN_MLX_TOY = True to run the MLX toy example.")

## 7. CLI 기준 실행 예시

노트북 밖에서는 아래 명령으로 동일한 경로를 실행할 수 있습니다.

```bash
diffhash mlx-conditional \
  --data-root notebooks/_demo_diffusion_data/data/images \
  --json-root notebooks/_demo_diffusion_data/output/json \
  --output-dir notebooks/_demo_diffusion_data/output/conditional_diffusion_mlx \
  --train-steps 1 \
  --timesteps 4 \
  --batch-size 2 \
  --image-size 8 \
  --hidden-dim 16 \
  --time-dim 8 \
  --device cpu
```

모듈 기준 실행도 가능합니다.

```bash
python -m diffusion_hash_inv.cli mlx-conditional --help
python -m diffusion_hash_inv.cli mlx-toy --help
```

## 8. 출력 확인

In [ ]:
for path in sorted((DEMO_ROOT / "output").rglob("*.png")):
    print(path.relative_to(DEMO_ROOT))